In [25]:
import requests as rq
import pandas as pd
import datetime as dt
from time import sleep

In [26]:
with open('accessToken.txt',"r") as file:
    access_token = file.read()

In [27]:
df = pd.read_csv("https://assets.upstox.com/market-quote/instruments/exchange/complete.csv.gz")

In [28]:
def filter_df(df,lot_size):
    df = df[(df['exchange'] == 'NSE_FO') &(df['instrument_type'] == 'OPTIDX') &(df['lot_size'] == lot_size)]
    df = df[df['expiry'] ==  min(df['expiry'].unique())]
    return df
BNDF = filter_df(df,15)

In [30]:
optionChain = list(BNDF['instrument_key'])

In [31]:
def get_quotes(instrument):
	url = 'https://api-v2.upstox.com/market-quote/ltp'
	headers = {
		'accept': 'application/json',
		'Api-Version': '2.0',
		'Authorization': f'Bearer {access_token}'}
	params = {'symbol': instrument,'interval':'1d'}
	res = rq.get(url, headers=headers, params=params).json()
	return res
ltp_data = get_quotes(optionChain)
ltp_data

{'status': 'success',
 'data': {'NSE_FO:BANKNIFTY2432048100PE': {'last_price': 1592.25,
   'instrument_token': 'NSE_FO|41490'},
  'NSE_FO:BANKNIFTY2432043800CE': {'last_price': 2597.8,
   'instrument_token': 'NSE_FO|41000'},
  'NSE_FO:BANKNIFTY2432044600CE': {'last_price': 2365.65,
   'instrument_token': 'NSE_FO|41095'},
  'NSE_FO:BANKNIFTY2432047300PE': {'last_price': 872.5,
   'instrument_token': 'NSE_FO|41408'},
  'NSE_FO:BANKNIFTY2432043000PE': {'last_price': 1.7,
   'instrument_token': 'NSE_FO|40971'},
  'NSE_FO:BANKNIFTY2432053500CE': {'last_price': 2.1,
   'instrument_token': 'NSE_FO|50708'},
  'NSE_FO:BANKNIFTY2432045400CE': {'last_price': 1178.6,
   'instrument_token': 'NSE_FO|41207'},
  'NSE_FO:BANKNIFTY2432050000CE': {'last_price': 3.15,
   'instrument_token': 'NSE_FO|41654'},
  'NSE_FO:BANKNIFTY2432048900CE': {'last_price': 8.65,
   'instrument_token': 'NSE_FO|41547'},
  'NSE_FO:BANKNIFTY2432050300PE': {'last_price': 3289.15,
   'instrument_token': 'NSE_FO|47652'},
  'NSE_F

In [35]:
def find_option(optionChain,near_val):
	call_symbol = {}
	put_symbol = {}
	trade_symbol = {}
	for i in range(5):
		try:
			ltp_data = get_quotes(optionChain)['data']
		except:
			sleep(0.5)
			ltp_data = get_quotes(optionChain)['data']
		for k,v in ltp_data.items():
			# Call Symbol
			if float(v['last_price']) <= near_val and k[-2:] == 'CE':
				call_symbol.update({v['instrument_token']: float(v['last_price'])})
			# Put Symbol
			if float(v['last_price']) <= near_val and k[-2:] == 'PE':
				put_symbol.update({v['instrument_token']: float(v['last_price'])})
		if call_symbol and put_symbol:
			ce_val = min(list(call_symbol.values()), key=lambda x: abs(x-near_val))
			pe_val = min(list(put_symbol.values()), key=lambda x: abs(x-near_val))
			for a, b in call_symbol.items():
				if b == ce_val:
					trade_symbol.update({a: b})
			for c, d in put_symbol.items():
				if d == pe_val:
					trade_symbol.update({c: d})
		if trade_symbol:
			return trade_symbol
		else:
			sleep(1)
	return 'Symbol Not found'
trade_symbol = find_option(optionChain,100)
trade_symbol

{'NSE_FO|41409': 83.5, 'NSE_FO|41297': 90.0}